# SmartHire — Unsupervised Job Clustering & Topic Discovery

Discovers natural job families using K-Means, evaluates optimal k with Elbow and Silhouette methods, and visualizes clusters with 2D PCA.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
from src.config import CLEAN_JOBS_PATH
from src.models.clustering import cluster_jobs, find_optimal_k, compute_pca_projection, extract_job_topics

jobs = pd.read_csv(CLEAN_JOBS_PATH)


### 1. Elbow Method & Silhouette Score


In [ ]:
elbow = find_optimal_k(jobs.head(1500), k_range=range(2, 7))
fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.plot(elbow['k_values'], elbow['inertias'], 'b-o')
ax2.plot(elbow['k_values'], elbow['silhouette_scores'], 'g-s')
ax1.set_xlabel('k (Number of Clusters)')
ax1.set_ylabel('Inertia', color='b')
ax2.set_ylabel('Silhouette Score', color='g')
plt.title('Optimal k Selection for Job Clusters')
plt.show()


### 2. K-Means Clustering & 2D PCA Visualization


In [ ]:
clustered, km, vec, info = cluster_jobs(jobs, max_rows=3000, save=True)
texts = clustered['job_text'].fillna('').astype(str)
mat = vec.transform(texts)
coords = compute_pca_projection(mat, n_components=2)

plt.figure(figsize=(9, 6))
plt.scatter(coords[:, 0], coords[:, 1], c=clustered['cluster'], cmap='tab10', alpha=0.5, s=15)
plt.title("2D PCA Projection of Job Clusters")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.colorbar(label='Cluster ID')
plt.show()


### 3. Latent Dirichlet Allocation (LDA) Topic Extraction


In [ ]:
topics = extract_job_topics(jobs.head(2000), n_topics=5)
for t in topics:
    print(f"Topic {t['topic_id']}: {t['theme']} -> Keywords: {', '.join(t['keywords'])}")
